# Shelvy — YOLOv8 Instance Segmentation (Bread + Mold)

**Google Colab / free T4 GPU**  
Dataset lives in **Google Drive** (Roboflow **Instance Segmentation** export, YOLOv8 format).

Run cells top to bottom. Each cell says what it does and what to check.

> Before you start: **Runtime -> Change runtime type -> T4 GPU**.

## 1. Confirm the GPU is on

In [ ]:
!nvidia-smi
# You should see 'Tesla T4' and ~15360MiB memory. If it errors, the GPU is off:
# Runtime -> Change runtime type -> T4 GPU, then re-run.

## 2. Install Ultralytics (YOLOv8)

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()  # prints version + confirms CUDA is available

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Point to your dataset in Drive

Put your Roboflow export folder anywhere in Drive, then set `DATASET_DIR` below to that folder.
It must contain `data.yaml` and `train/ valid/ (test/)` subfolders.

Example: if it's in `MyDrive/Shelvy/mold-seg-v3`, set:
`DATASET_DIR = '/content/drive/MyDrive/Shelvy/mold-seg-v3'`

In [ ]:
import os

# <<< EDIT THIS ONE LINE to match your folder in Drive >>>
DATASET_DIR = '/content/drive/MyDrive/Shelvy/mold-seg-v3'

assert os.path.isdir(DATASET_DIR), f'Not found: {DATASET_DIR}  (check the path / spelling)'
print('Contents:', os.listdir(DATASET_DIR))
assert 'data.yaml' in os.listdir(DATASET_DIR), 'No data.yaml here - is this the Roboflow export folder?'

## 5. Fix the data.yaml paths to absolute

Roboflow writes relative paths (`../train/images`) that break in Colab. This rewrites them to absolute
paths and prints your class names so you can confirm order (must be Bread, Mold).

In [ ]:
import yaml

yaml_path = os.path.join(DATASET_DIR, 'data.yaml')
with open(yaml_path) as f:
    data = yaml.safe_load(f)

data['path']  = DATASET_DIR
data['train'] = os.path.join(DATASET_DIR, 'train', 'images')
data['val']   = os.path.join(DATASET_DIR, 'valid', 'images')
if os.path.isdir(os.path.join(DATASET_DIR, 'test', 'images')):
    data['test'] = os.path.join(DATASET_DIR, 'test', 'images')

with open(yaml_path, 'w') as f:
    yaml.safe_dump(data, f, sort_keys=False)

print('Classes (index order matters):', data['names'])
print('nc =', data.get('nc'))
print('train ->', data['train'])
print('val   ->', data['val'])

## 6. Check the class balance BEFORE training

This counts how many *instances* of each class exist in train + valid. This is the honest answer to
"is my dataset balanced?" - if Mold hugely outnumbers Bread (or vice-versa), you'll see it here.

In [ ]:
import glob
from collections import Counter

names = data['names'] if isinstance(data['names'], list) else [data['names'][i] for i in sorted(data['names'])]

def count_instances(split):
    c = Counter()
    imgs = 0
    for lbl in glob.glob(os.path.join(DATASET_DIR, split, 'labels', '*.txt')):
        imgs += 1
        with open(lbl) as f:
            for line in f:
                line = line.strip()
                if line:
                    c[int(line.split()[0])] += 1
    return imgs, c

for split in ['train', 'valid']:
    imgs, c = count_instances(split)
    print(f'\n[{split}]  images: {imgs}')
    total = sum(c.values()) or 1
    for i, nm in enumerate(names):
        print(f'  {nm:<8} {c.get(i,0):>5} instances  ({100*c.get(i,0)/total:.1f}%)')

## 7. Train

Settings chosen for a SMALL, IMBALANCED mold dataset on a free T4:
- `yolov8s-seg` = good accuracy/speed on T4. (Try `yolov8m-seg` only if you have GPU budget.)
- `epochs=200`, `patience=25` -> trains long, auto-stops when it stops improving.
- `imgsz=768` -> keeps small mold spots big enough to learn (BIGGEST lever for small objects).
- `copy_paste=0.3` + `mosaic` -> pastes/mixes instances = fights class imbalance for segmentation.
- `flipud=0.5` -> mold has no 'up', so vertical flips are free extra data.
- If you hit CUDA out-of-memory: lower `batch` to 8, or `imgsz` to 640.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s-seg.pt')  # pretrained COCO seg weights

results = model.train(
    data=yaml_path,
    epochs=200,
    patience=25,
    imgsz=768,
    batch=12,            # T4-safe at imgsz=768; use 16 at imgsz=640; -1 for auto
    device=0,
    project='/content/runs',
    name='shelvy_seg',
    # --- augmentation tuned for small + imbalanced ---
    mosaic=1.0,
    close_mosaic=15,     # turn mosaic off for last 15 epochs = cleaner fine-tune
    copy_paste=0.3,
    fliplr=0.5,
    flipud=0.5,
    degrees=10,
    scale=0.5,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,  # handles chamber lighting changes
    plots=True,
)

## 8. Read the REAL metrics (per class)

This prints the mask mAP overall AND per class, so you can see if it's Bread or Mold dragging the score.
For segmentation, look at **`mAP50-95(M)`** = the mask metric. 0.50 there is actually decent; 0.50 on
`mAP50` is the weak one.

In [ ]:
metrics = model.val()
print('\n=== MASK metrics (segmentation) ===')
print('mAP50-95(M):', round(metrics.seg.map, 4))
print('mAP50   (M):', round(metrics.seg.map50, 4))
print('\nPer-class mask mAP50-95:')
for i, nm in enumerate(names):
    try:
        print(f'  {nm:<8} {metrics.seg.maps[i]:.4f}')
    except Exception:
        pass

## 9. Look at predictions on the validation images

Numbers lie less than pictures. This runs the best weights on a few val images so you can eyeball
whether the masks actually land on the mold.

In [ ]:
from IPython.display import Image, display

best = '/content/runs/shelvy_seg/weights/best.pt'
val_images = os.path.join(DATASET_DIR, 'valid', 'images')
pred = YOLO(best)
pred.predict(source=val_images, imgsz=768, conf=0.25, save=True,
             project='/content/runs', name='shelvy_pred', exist_ok=True)

shots = glob.glob('/content/runs/shelvy_pred/*.jpg')[:6]
for s in shots:
    display(Image(filename=s, width=520))

## 10. Save the trained weights back to Drive

Colab wipes /content when the session ends. This copies `best.pt` (and the training plots) into Drive
so you keep them.

In [ ]:
import shutil

out = '/content/drive/MyDrive/Shelvy/trained'
os.makedirs(out, exist_ok=True)
shutil.copy('/content/runs/shelvy_seg/weights/best.pt', os.path.join(out, 'shelvy_seg_best.pt'))
for plot in ['results.png', 'confusion_matrix.png', 'BoxPR_curve.png', 'MaskPR_curve.png']:
    p = os.path.join('/content/runs/shelvy_seg', plot)
    if os.path.exists(p):
        shutil.copy(p, os.path.join(out, plot))
print('Saved to', out)
print(os.listdir(out))

## 11. (OPTIONAL) Hyperparameter tuning — only if you have GPU budget left

`model.tune()` runs MANY short trainings to search hyperparameters. On a free T4 this can take hours
and eat your quota. **Do this only after** you have a good baseline from step 7 and time to spare.
Leave it commented out otherwise.

In [ ]:
# from ultralytics import YOLO
# tuner = YOLO('yolov8s-seg.pt')
# tuner.tune(data=yaml_path, epochs=30, iterations=20, imgsz=768,
#            optimizer='AdamW', plots=False, save=False, val=True)